# Chapter 03 - Inheritance and `super()`

#### 1. Mechanical Refresher
A subclass inherits attributes and methods from its parent class, can override a parent method by defining a method with the same name, and can call the parent implementation through `super()`. Method lookup checks the instance's class first, then parent classes in method-resolution order, and composition means storing another object and delegating work to it instead of inheriting from it.

#### 2. Minimal Working Example

In [ ]:
class Report:
    def __init__(self, name):
        self.name = name

    def label(self):
        return "report:" + self.name

class TimedReport(Report):
    def __init__(self, name, seconds):
        super().__init__(name)
        self.seconds = seconds

    def label(self):
        return super().label() + " in " + str(self.seconds) + "s"

print(TimedReport("train", 8).label())

`TimedReport` gets `name` setup from `Report.__init__` through `super()`, then adds `seconds`. Its `label` method overrides the parent method, calls the parent version, and extends the returned string.

#### 3. Modify Drills

**Modify Drill 1.** Change the subclass label suffix and predict the final string.

In [ ]:
class Label:
    def text(self):
        return "base"

class LoudLabel(Label):
    def text(self):
        return super().text() + "!"

actual = LoudLabel().text()
expected = "base!"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Modify Drill 2.** Add a subclass-only attribute through `super().__init__` plus extra assignment.

In [ ]:
class Run:
    def __init__(self, name):
        self.name = name

class ScoredRun(Run):
    def __init__(self, name, score):
        super().__init__(name)
        self.score = score

run = ScoredRun("trial", 0.8)
actual = [run.name, run.score]
expected = ["trial", 0.8]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Modify Drill 3.** Prefer composition when the new object only uses another object's service.

In [ ]:
class Formatter:
    def format(self, value):
        return "[" + str(value) + "]"

class Printer:
    def __init__(self, formatter):
        self.formatter = formatter

    def preview(self, value):
        return self.formatter.format(value)

actual = Printer(Formatter()).preview(7)
expected = "[7]"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 4. Break-and-Fix Drills

**Break-and-Fix Drill 1.** Break it by deleting `super().__init__(name)`. Predict why `self.name` is missing, then restore the parent initialization call.

In [ ]:
class Named:
    def __init__(self, name):
        self.name = name

class Tagged(Named):
    def __init__(self, name, tag):
        super().__init__(name)
        self.tag = tag

    def label(self):
        return self.tag + ":" + self.name

actual = Tagged("alpha", "run").label()
expected = "run:alpha"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Break-and-Fix Drill 2.** Break it by replacing `super().value()` with `self.value()`. Predict the infinite recursion, then call the parent version with `super()`.

In [ ]:
class BaseValue:
    def value(self):
        return 10

class OffsetValue(BaseValue):
    def value(self):
        return super().value() + 5

actual = OffsetValue().value()
expected = 15
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 5. Self-Verification
Expected-vs-actual prints remain the verification mechanism until the `assert` chapter. In break-and-fix drills, verify both the fixed value and your prediction of the broken symptom.

#### 6. Standalone Exercises

**Exercise 1.** Make `Dog.speak` override `Animal.speak`. Expected behavior: `'woof'`.

In [ ]:
class Animal:
    def speak(self):
        return "sound"

class Dog(Animal):
    # TODO: define speak.
    pass

actual = Dog().speak()
expected = "woof"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Exercise 2.** Use `super()` so the subclass keeps the parent setup. Expected behavior: `['run', 3]`.

In [ ]:
class NamedItem:
    def __init__(self, name):
        self.name = name

class CountedItem(NamedItem):
    def __init__(self, name, count):
        # TODO: call parent setup, then store count.
        self.count = count

item = CountedItem("run", 3)
actual = [item.name, item.count]
expected = ["run", 3]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Exercise 3.** Override and extend a parent method. Expected behavior: `'value=5; unit=kg'`.

In [ ]:
class Measurement:
    def describe(self):
        return "value=5"

class WeightedMeasurement(Measurement):
    def describe(self):
        # TODO: extend the parent string.
        return ""

actual = WeightedMeasurement().describe()
expected = "value=5; unit=kg"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Exercise 4.** Use composition to reuse `Tokenizer` without inheriting from it. Expected behavior: `['a', 'b']`.

In [ ]:
class Tokenizer:
    def split(self, text):
        return text.split("-")

class Parser:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def parse(self, text):
        # TODO: delegate to self.tokenizer.
        return None

actual = Parser(Tokenizer()).parse("a-b")
expected = ["a", "b"]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Exercise 5.** Build a two-level inheritance chain and predict lookup. Expected behavior: `'child-parent-root'`.

In [ ]:
class Root:
    def name(self):
        return "root"

class Parent(Root):
    def name(self):
        return "parent-" + super().name()

class Child(Parent):
    def name(self):
        return None  # TODO: prefix child- to parent result.

actual = Child().name()
expected = "child-parent-root"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 7. Applied AI/ML Drill
**ML to Python mirror:** a specialized tracker subclass is just an inherited class that reuses parent storage and overrides the part that formats the result. **Python to ML mirror:** model and callback libraries use the same pattern when a base class supplies shared bookkeeping and subclasses customize one method such as `step`, `forward`, or `on_epoch_end`.

**Applied Drill.** Complete `LossTracker.summary` by extending the parent summary. Expected behavior: `'tracker:loss last=0.25'`.

In [ ]:
class MetricTracker:
    def __init__(self, name):
        self.name = name
        self.values = []

    def add(self, value):
        self.values.append(value)

    def summary(self):
        return "tracker:" + self.name

class LossTracker(MetricTracker):
    def summary(self):
        # TODO: call parent summary and add the last value.
        return ""

tracker = LossTracker("loss")
tracker.add(0.25)
actual = tracker.summary()
expected = "tracker:loss last=0.25"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 8. Common Bugs
- Skipping `super().__init__`: parent attributes never get created; the symptom is usually `AttributeError` later.
- Calling `self.method()` inside the overriding method when you meant `super().method()`: the symptom is infinite recursion.
- Inheriting only to reuse one helper method: the symptom is a confusing class hierarchy; composition is usually clearer.
- Assuming a parent method edits subclass state automatically: `super()` calls exactly the parent method you ask for; it does not guess extra subclass behavior.

#### 9. Compounding Drill

Combine inheritance with Chapter 1 function objects: make a subclass that stores a metric function and extends a parent report.

In [ ]:
def absolute_error(pred, target):
    return abs(pred - target)

class Report:
    def __init__(self, name):
        self.name = name

    def line(self):
        return self.name

class MetricReport(Report):
    def __init__(self, name, metric_fn):
        super().__init__(name)
        self.metric_fn = metric_fn

    def line(self, pred, target):
        # TODO: include parent line and metric result.
        return ""

actual = MetricReport("eval", absolute_error).line(9, 4)
expected = "eval:5"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 10. Chapter Project
**Goal:** Build a tiny metric-reporting system using functions as objects, instance state, inheritance, and `super()`.

**Requirements:** create a base `MetricReport` with `name` and `metric_fn`; create one subclass that adds a `split` label; create an `evaluate(pred, target)` method that returns a readable string.

**Stretch Goals:** add a second subclass for percentage formatting; add a composition-based formatter object and compare the design.

**Evaluation Checklist:** separate instances do not share state; subclass setup calls `super()`; the metric function is passed in, not hard-coded; expected-vs-actual output confirms at least two metrics.

In [ ]:
def abs_error(pred, target):
    return abs(pred - target)

class MetricReport:
    def __init__(self, name, metric_fn):
        self.name = name
        self.metric_fn = metric_fn

    def evaluate(self, pred, target):
        return self.name + "=" + str(self.metric_fn(pred, target))

class SplitMetricReport(MetricReport):
    def __init__(self, name, metric_fn, split):
        # TODO: call parent setup, then store split.
        self.split = split

    def evaluate(self, pred, target):
        # TODO: prefix the parent output with split.
        return ""

actual = SplitMetricReport("abs_error", abs_error, "train").evaluate(5, 2)
expected = "train:abs_error=3"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 11. Solutions Appendix

--- SOLUTIONS: DO NOT PEEK UNTIL ATTEMPTED ---

In [ ]:
# Standalone Exercises
class Animal:
    def speak(self):
        return "sound"

class Dog(Animal):
    def speak(self):
        return "woof"

print(Dog().speak())

class NamedItem:
    def __init__(self, name):
        self.name = name

class CountedItem(NamedItem):
    def __init__(self, name, count):
        super().__init__(name)
        self.count = count

item = CountedItem("run", 3)
print([item.name, item.count])

In [ ]:
class Measurement:
    def describe(self):
        return "value=5"

class WeightedMeasurement(Measurement):
    def describe(self):
        return super().describe() + "; unit=kg"

print(WeightedMeasurement().describe())

class Tokenizer:
    def split(self, text):
        return text.split("-")

class Parser:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def parse(self, text):
        return self.tokenizer.split(text)

print(Parser(Tokenizer()).parse("a-b"))

In [ ]:
class Root:
    def name(self):
        return "root"

class Parent(Root):
    def name(self):
        return "parent-" + super().name()

class Child(Parent):
    def name(self):
        return "child-" + super().name()

print(Child().name())

class MetricTracker:
    def __init__(self, name):
        self.name = name
        self.values = []

    def add(self, value):
        self.values.append(value)

    def summary(self):
        return "tracker:" + self.name

class LossTracker(MetricTracker):
    def summary(self):
        return super().summary() + " last=" + str(self.values[-1])

tracker = LossTracker("loss")
tracker.add(0.25)
print(tracker.summary())

In [ ]:
# Compounding Drill and Chapter Project approach
def absolute_error(pred, target):
    return abs(pred - target)

class Report:
    def __init__(self, name):
        self.name = name

    def line(self):
        return self.name

class MetricReportLine(Report):
    def __init__(self, name, metric_fn):
        super().__init__(name)
        self.metric_fn = metric_fn

    def line(self, pred, target):
        return super().line() + ":" + str(self.metric_fn(pred, target))

print(MetricReportLine("eval", absolute_error).line(9, 4))

class MetricReport:
    def __init__(self, name, metric_fn):
        self.name = name
        self.metric_fn = metric_fn

    def evaluate(self, pred, target):
        return self.name + "=" + str(self.metric_fn(pred, target))

class SplitMetricReport(MetricReport):
    def __init__(self, name, metric_fn, split):
        super().__init__(name, metric_fn)
        self.split = split

    def evaluate(self, pred, target):
        return self.split + ":" + super().evaluate(pred, target)

print(SplitMetricReport("abs_error", absolute_error, "train").evaluate(5, 2))